# Step 3a: Simple baselines

Reference rankers that need no training. Each one scores every candidate in every impression, and candidates are ranked within their own impression. They establish what exposure counts, the candidate's own clicks, and related-content clicks achieve on their own, before any learned model.

| `model_id` | Score | What it establishes |
|---|---|---|
| `random` | constant score, so order comes only from the tie-break key | The floor any ranking must beat |
| `popularity` | `prior_exposures` | Ranking by how often the article was shown before. A reference method, not a reconstruction of MSN's ranker. |
| `category_ctr` | `category_prior_ctr` = prior CTR of other articles in the same category, shrunk toward the global CTR (50 pseudo-exposures) | Related-content behaviour alone. Content features must beat this to justify themselves. |
| `candidate_smoothed_ctr` | (prior_clicks + β·g<sub>t</sub>) / (prior_exposures + β) | The candidate's own clicks with generic smoothing toward the global CTR g<sub>t</sub> |
| `content_smoothed_ctr` | (prior_clicks + α·related_ctr) / (prior_exposures + α) | The same smoothing, but toward a content-specific prior (subcategory → category → global) |

**Information cutoff.** Every input column is computed from events strictly before the impression's timestamp (feature schema contract v0.1). Counters keep updating through val and test; nothing here is fitted.

**Zero-support handling.** An article with no prior exposures gets popularity 0, candidate smoothed CTR = g<sub>t</sub>, and content smoothed CTR = related_ctr. A category with no earlier events gets g<sub>t</sub>. Ties (common for popularity) are broken by a seeded hash of `row_id` that ignores labels and is the same for every model.

**Tuning.** β and α are chosen on **validation only**, over the same grid {5, 10, 20, 50, 100}. Selection rule (shared with steps 4 and 5): highest validation Cold Recall@10 among settings within 0.005 validation nDCG@10 of the best, ties broken by nDCG@10. The tolerance is provisional until the team agrees it. The test split is scored once, at the end, with the frozen choices.

**Controlled comparison (handover A3).** `candidate_smoothed_ctr` and `content_smoothed_ctr` are compared at the **same strength** (α = β) for every grid value, and also at their own best values. Same candidates, splits, slates and cutoffs throughout.

**Outputs.** One prediction file per model, config and split with columns `row_id, model_id, config_id, predicted_score`, for Shi Ting's evaluator.

> Metrics in this notebook come from `shared/eval_interim.py`, a stand-in used only for tuning until the team evaluator exists. The numbers reported in the final report come from `step6_evaluation_diagnostics.ipynb`.

## 1. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.getcwd()
    DATA_ROOT = os.path.join(REPO_DIR, "data")
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from shared import schema as S, derived as D, selection as SEL
from shared.predictions import write_predictions
from shared import eval_interim as E      # swap for the team evaluator once it exists

# REAL for submission runs. SYNTHETIC only for local development before real features exist.
DATA_KIND = os.environ.get("G11_DATA_KIND", "REAL")
if DATA_KIND == "REAL":
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
else:
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"synthetic_v{S.SCHEMA_VERSION}", "learnable")
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")

version = S.read_version(FEATURE_ROOT)
assert version.get("data_kind", "REAL") == DATA_KIND, f"feature tables are {version.get('data_kind')}, run is {DATA_KIND}"
print(f"DATA_KIND = {DATA_KIND} | features: {FEATURE_ROOT} | schema {version['schema_version']}")

## 2. Load only the columns the baselines use

Labels and cold flags come from the metadata table; scores come from the feature table. The two are aligned on `row_id`.

In [ ]:
META_COLS = ["row_id", "impression_id", "label", "is_cold_10", "is_zero_exposure"]
FEAT_COLS = ["row_id", "prior_exposures", "prior_clicks", "global_prior_ctr", "category_prior_ctr", "related_ctr"]

data = {}
for split in S.SPLITS:
    m = pd.read_parquet(S.meta_path(FEATURE_ROOT, split), columns=META_COLS)
    f = pd.read_parquet(S.features_path(FEATURE_ROOT, split), columns=FEAT_COLS)
    assert np.array_equal(m["row_id"].to_numpy(), f["row_id"].to_numpy()), f"{split}: row_id misaligned"
    data[split] = (m, f)

pd.DataFrame({split: {"impressions": m["impression_id"].nunique(), "candidate rows": len(m),
                      "CTR": m["label"].mean(), "cold row share (<10)": m["is_cold_10"].mean(),
                      "clicked cold candidates": int((m["label"] * m["is_cold_10"]).sum())}
              for split, (m, f) in data.items()}).T.round(4)

## 3. Scoring functions

In [ ]:
GRID = S.ALPHA_GRID
TUNED = ("candidate_smoothed_ctr", "content_smoothed_ctr")

def score(model_id, f, strength=None):
    if model_id == "random":
        return np.zeros(len(f))
    if model_id == "popularity":
        return f["prior_exposures"].to_numpy(np.float64)
    if model_id == "category_ctr":
        return f["category_prior_ctr"].to_numpy(np.float64)
    if model_id == "candidate_smoothed_ctr":
        return D.candidate_smoothed_ctr(f, strength)
    if model_id == "content_smoothed_ctr":
        return D.content_smoothed_ctr(f, strength)
    raise ValueError(model_id)

def config_id(model_id, strength):
    if model_id == "candidate_smoothed_ctr":
        return f"beta={strength}"
    if model_id == "content_smoothed_ctr":
        return f"alpha={strength}"
    return "default"

SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10", "n_cold_click_impressions"]

## 4. Tune β and α on validation

Every grid value is scored on validation, and the prediction file is saved, so the team evaluator can reproduce the selection later.

In [ ]:
val_meta, val_feat = data["val"]
val_per_imp, rows = {}, []
for model_id in TUNED:
    for s in GRID:
        sc = score(model_id, val_feat, s)
        write_predictions(val_meta["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=model_id,
                          config_id=config_id(model_id, s), split="val", schema_version=S.SCHEMA_VERSION)
        pi = E.per_impression_metrics(val_meta, sc)
        val_per_imp[(model_id, s)] = pi
        rows.append({"model_id": model_id, "strength": s, **E.summarise(pi)})
grid = pd.DataFrame(rows).set_index(["model_id", "strength"])
grid[SHOW].round(4)

In [ ]:
SELECT = SEL.PRIMARY
print("selection rule:", SEL.describe())
best = {}
for model_id in TUNED:
    g = grid.loc[model_id]
    top = SEL.select(g)
    best[model_id] = int(top)
    print(f"{model_id}: selected strength {top} "
          f"(val {SEL.PRIMARY} = {g.loc[top, SEL.PRIMARY]:.4f}, {SEL.GUARD} = {g.loc[top, SEL.GUARD]:.4f})")

## 5. Controlled comparison: content-specific prior vs global prior

For each strength s, compare `content_smoothed_ctr` (α = s) with `candidate_smoothed_ctr` (β = s) on the **same validation impressions**. The difference is content minus global. The 95% interval is a paired bootstrap over impressions (1,000 resamples), and the last column says how many impressions with a cold click support each Cold Recall@10 comparison.

A positive, stable difference with an interval above zero supports the conclusion: *better performance under these controlled conditions supports the usefulness of the content-specific prior for this task.* A difference whose interval crosses zero does not.

In [ ]:
cmp_rows = []
pairs = [(f"alpha=beta={s}", s, s) for s in GRID]
pairs.append((f"best vs best (alpha={best['content_smoothed_ctr']}, beta={best['candidate_smoothed_ctr']})",
              best["content_smoothed_ctr"], best["candidate_smoothed_ctr"]))
for label, a, b in pairs:
    pc, pg = val_per_imp[("content_smoothed_ctr", a)], val_per_imp[("candidate_smoothed_ctr", b)]
    row = {"comparison": label}
    for metric in ("ndcg@10", "auc", "mrr"):
        row[f"Δ {metric}"] = pc[metric].mean() - pg[metric].mean()
    bs = E.paired_bootstrap(pg["cold_recall@10"], pc["cold_recall@10"])
    row.update({"Δ cold_recall@10": bs["mean_diff"], "95% CI low": bs["ci_low"], "95% CI high": bs["ci_high"],
                "cold-click impressions": bs["n_impressions"]})
    cmp_rows.append(row)
comparison = pd.DataFrame(cmp_rows).set_index("comparison")
comparison.round(4)

## 6. Final scoring with frozen choices

The selections above are now fixed. Every baseline is scored on train, validation and test, and its prediction files are written. Train and validation are shown because the report must give training and testing performance; this is the first and only time the test split is scored.

In [ ]:
FINAL = [("random", None), ("popularity", None), ("category_ctr", None),
         ("candidate_smoothed_ctr", best["candidate_smoothed_ctr"]),
         ("content_smoothed_ctr", best["content_smoothed_ctr"])]

final_rows, paths = [], []
for split in S.SPLITS:
    meta, feat = data[split]
    for model_id, s in FINAL:
        sc = score(model_id, feat, s)
        cfg = config_id(model_id, s)
        paths.append(write_predictions(meta["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=model_id,
                                       config_id=cfg, split=split, schema_version=S.SCHEMA_VERSION,
                                       extra={"selected_on": "val", "selection_metric": SELECT}))
        final_rows.append({"split": split, "model_id": model_id, "config_id": cfg, **E.evaluate(meta, sc)})
final = pd.DataFrame(final_rows).set_index(["split", "model_id"])
print(f"wrote {len(paths)} prediction files under {os.path.join(PRED_ROOT, DATA_KIND.lower())}")
final[["config_id"] + SHOW].round(4)

## 7. Reading the results

* `random` is the floor. Any baseline that does not clear it on a metric carries no signal for that metric.
* `popularity` can only separate articles that already have exposure history. Every zero-exposure candidate gets the same score, so its Cold Recall reflects how few cold articles it lifts into the top 10.
* `category_ctr` gives the floor that content features in the advanced model must beat.
* The controlled comparison in section 5 is the evidence for or against the content-specific prior. Its conclusion should be stated only as strongly as the confidence interval and the number of cold-click impressions allow.